## Distiller SimSwap 512 pour remplacer son générateur

Exécuter `dataset_prep.ipynb` avant ce notebook. L'élève reçoit exactement
`(X0, E1)` : RGB float32 NCHW en `[0, 1]`, crop ffhq 512 et embedding **SimSwap**
L2 de dimension 512. Il renvoie une image RGB de même forme en `[0, 1]`.
`X1` est le portrait donneur permettant de calculer E1, pas une deuxième entrée image
du générateur. Le backend continuera d'assurer détection, alignement, ArcFace et blending.

L'élève conserve l'architecture officielle, avec 8 blocs AdaIN au lieu de 9.
Les couches correspondantes sont initialisées avec les poids du maître ; seul son
dernier bloc résiduel est retiré. La réduction de paramètres est mesurée plus bas.
Le discriminateur sert uniquement à l'entraînement et n'est pas exporté pour l'inférence.

Les cibles sont les sorties Y pré-calculées, sans BiSeNet, quantifiées en PNG RGB uint8 sur disque.
Le dataset les recharge en float32 CHW `[0, 1]`, comme X0 et X1. La perte de reconstruction
compare l'élève à Y, jamais à X0. La validation suit des X0 absents de l'entraînement ;
elle ne mesure pas encore la généralisation à de nouveaux donneurs. La convergence
et la qualité doivent être vérifiées sur les courbes et les images, pas supposées.

**Reprise :** exécuter les cellules de préparation du GAN, y compris le résumé et
`CHECKPOINTS COMPLETS ET REPRISE`, puis l'entraînement. `resume_training = True`
recharge automatiquement `outputs/gan_maison/training_last.pth`.
Le checkpoint comprend G, D, Adam, scheduler, époque/batch, ordre des données,
historique, early stopping et états aléatoires Python/NumPy/PyTorch/CUDA.
Il est écrit atomiquement tous les 500 steps, avant la validation et en fin d'époque.
Après interruption, seuls les steps depuis la dernière sauvegarde sont à rejouer
(environ 500 en cours de train) ; une validation interrompue est recommencée.
Le dataset reste sur disque et les batches train déjà sauvegardés ne sont pas rechargés.
Les meilleurs poids d'inférence restent dans `lite_generator_best.pth`.
Ne pas changer le batch size, le manifeste ou le warmup pour reprendre le même run.
`num_epochs` peut être augmenté pour prolonger un entraînement terminé.
Les anciennes sauvegardes contenant uniquement les poids de l'élève ne permettent
pas de reconstruire rétroactivement les états manquants du discriminateur et d'Adam.

**Chargement optimise :** X1 est lu uniquement pour l'aperçu ; le train utilise X0, E1 et Y.
Les embeddings E1 sont mis en cache en RAM. Sur CUDA, les operations CPU utilisent
4 threads et un thread prepare le batch suivant pendant le calcul GPU, sans reordonner
les exemples. Les copies utilisent la memoire epinglee du DataLoader.
Le dataset reste sans augmentation aleatoire ; un dataset declare non deterministe
desactive le prechargement pour proteger la reprise RNG. L'architecture, le batch size,
les pertes, la precision float32, les checkpoints et l'early stopping restent identiques.

In [1]:
# --- 1. IMPORTS ET REPRODUCTIBILITÉ ---
import gc
import json
import random
import logging
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from src.utils.config import Settings
from src.utils.image_utils import load_image
from src.utils.model_manager import sha256
from src.backends.simswap_backend import SimSwapBackend, add_official_source

logging.getLogger().setLevel(logging.ERROR)
settings = Settings(device="cuda" if torch.cuda.is_available() else "cpu",
                    crop_size=512, segmentation=False)
device = settings.device
if device == "cuda":
    torch.set_num_threads(4)  # Eviter trop de threads pour les petites operations CPU
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
add_official_source(settings.simswap)
from models.fs_networks_512 import Generator_Adain_Upsample
print(f"L'entraînement s'effectuera sur : {device.upper()}")

L'entraînement s'effectuera sur : CUDA


In [2]:
# --- 2. ARCHITECTURE DU GÉNÉRATEUR ALLÉGÉ (L'ÉLÈVE) ---
class CustomLiteGenerator(Generator_Adain_Upsample):
    def __init__(self, style_dim=512, num_blocks=8):
        super().__init__(input_nc=3, output_nc=3, latent_size=style_dim,
                         n_blocks=num_blocks, deep=False)
        self.style_dim = style_dim
        self.num_blocks = num_blocks
    # Le forward officiel conserve (tanh(x) + 1) / 2 : sortie RGB [0, 1].

# --- ARCHITECTURE DU DISCRIMINATEUR CONDITIONNÉ PAR X0 ---
class PatchDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(6, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(256),
            nn.LeakyReLU(0.2),
            nn.Conv2d(256, 1, kernel_size=4, padding=1)
        )

    def forward(self, x0, image):
        # Discrimination à 256 pixels pour limiter la mémoire à l'entraînement.
        return self.model(F.avg_pool2d(torch.cat((x0, image), dim=1), 2))

In [3]:
# --- 3. CHARGEMENT DES TRIPLETS X0, X1, E1, Y ---
dataset_dir = Path("processed_dataset_512/distillation")
manifest_path = dataset_dir / "manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
contract = manifest["configuration"]
if (contract["version"] != 1 or contract["crop_size"] != 512
        or contract["alignment"] != "ffhq" or contract["color"] != "RGB"
        or contract["image_range"] != [0, 1]):
    raise ValueError("Le dataset ne respecte pas le contrat SimSwap 512")
if contract["preprocessing_hashes"]["src/backends/simswap_backend.py"] != sha256(
        settings.root / "src/backends/simswap_backend.py"):
    raise ValueError("Le backend de référence a changé depuis la préparation du dataset")

class FaceSwapDataset(Dataset):
    deterministic_loading = True  # PNG et embeddings fixes, sans augmentation aleatoire

    def __init__(self, root, records):
        self.root = Path(root)
        self.records = records
        self.identities = {}
        if not self.records:
            raise ValueError("Split vide : relancer la préparation du dataset")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        record = self.records[idx]
        def image_tensor(key):
            rgb = load_image(self.root / record[key])[..., ::-1]
            return torch.from_numpy(rgb.transpose(2, 0, 1).copy()).float() / 255
        x0 = image_tensor("X0")
        # E1 reste float32 ; quelques donneurs seulement, donc cache tres petit.
        identity = self.identities.get(record["E1"])
        if identity is None:
            identity = torch.from_numpy(np.load(self.root / record["E1"], allow_pickle=False).copy())
            if (identity.dtype != torch.float32 or identity.shape != (512,)
                    or not torch.isfinite(identity).all()
                    or not torch.isclose(identity.norm(), torch.tensor(1.), atol=1e-4)):
                raise ValueError(f"Identite SimSwap invalide : {record['key']}")
            self.identities[record["E1"]] = identity
        # PNG uint8 sur disque, RGB float32 CHW [0, 1] pour le réseau.
        y_path = self.root / record["Y"]
        png_target = y_path.suffix.lower() == ".png" or y_path.with_suffix(".png").exists()
        if y_path.suffix.lower() == ".png":
            y = image_tensor("Y")
        elif y_path.with_suffix(".png").exists():
            # Compatibilité avec un ancien manifeste pendant la migration.
            rgb = load_image(y_path.with_suffix(".png"))[..., ::-1]
            y = torch.from_numpy(rgb.transpose(2, 0, 1).copy()).float() / 255
        else:
            y = torch.from_numpy(np.load(y_path, allow_pickle=False).copy())
        if any(image.shape != (3, 512, 512) for image in (x0, y)):
            raise ValueError(f"Dimensions invalides : {record['key']}")
        # Un PNG uint8 / 255 est deja fini et borne ; garder les controles pour les anciens NPY.
        if y.dtype != torch.float32 or (not png_target and (
                not torch.isfinite(y).all() or y.min() < 0 or y.max() > 1)):
            raise ValueError(f"Cible Y invalide : {record['key']}")
        return x0, identity, y

def iter_prepared_batches(loader):
    # Un seul batch en avance, dans le meme ordre. Threads compatibles avec Jupyter/Windows.
    # Un dataset avec augmentation aleatoire reste synchrone pour conserver ses etats RNG.
    if device != "cuda" or not getattr(loader.dataset, "deterministic_loading", False):
        yield from loader
        return
    from concurrent.futures import ThreadPoolExecutor
    iterator = iter(loader)
    sentinel = object()
    executor = ThreadPoolExecutor(max_workers=1, thread_name_prefix="prepare-batch")
    future = None
    try:
        future = executor.submit(next, iterator, sentinel)
        while True:
            batch = future.result()
            if batch is sentinel:
                break
            future = executor.submit(next, iterator, sentinel)
            yield batch
    finally:
        if future is not None:
            future.cancel()
        executor.shutdown(wait=True, cancel_futures=True)

train_records = [row for row in manifest["records"] if row["split"] == "train"]
val_records = [row for row in manifest["records"] if row["split"] == "validation"]
if {row["source_id"] for row in train_records} & {row["source_id"] for row in val_records}:
    raise ValueError("Une image X0 apparaît dans les deux splits")
train_dataset = FaceSwapDataset(dataset_dir, train_records)
val_dataset = FaceSwapDataset(dataset_dir, val_records)
batch_size = 2  # Réduire à 1 si la VRAM est insuffisante
dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0,
                        pin_memory=device == "cuda", generator=torch.Generator().manual_seed(42))
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0,
                        pin_memory=device == "cuda")
print(f"Dataset prêt : {len(train_dataset)} exemples train, {len(val_dataset)} validation.")

Dataset prêt : 47992 exemples train, 11998 validation.


In [4]:
# --- 4. INITIALISATION DE L'ÉLÈVE À PARTIR DU MAÎTRE OFFICIEL ---
teacher = SimSwapBackend(settings)  # Chargement strict des poids actifs via le backend existant
from src.utils.model_manager import ModelManager
manager = ModelManager(settings)
for key in ("simswap512", "arcface"):
    if sha256(manager.require(key)) != contract["model_hashes"][key]:
        raise ValueError(f"Le checkpoint {key} diffère de celui qui a produit le dataset")

student_model = CustomLiteGenerator(num_blocks=8)
expected_keys = student_model.state_dict()
inherited = {key: value for key, value in teacher.generator.state_dict().items()
             if key in expected_keys}
student_model.load_state_dict(inherited, strict=True)
student_model = student_model.to(device)
teacher_params = sum(p.numel() for p in teacher.generator.parameters())
student_params = sum(p.numel() for p in student_model.parameters())
assert student_params < teacher_params
print(f"Maître : {teacher_params:,} paramètres ; élève : {student_params:,}")
print(f"Réduction du générateur : {100 * (1 - student_params / teacher_params):.2f} %")

# Garder seulement ArcFace pour la perte d'identité, avec gradient vers l'image générée.
arcface_model = teacher.arcface.eval().requires_grad_(False)
arc_mean, arc_std = teacher.mean, teacher.std
del expected_keys, inherited, teacher
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
discriminator = PatchDiscriminator().to(device)

Maître : 55,079,427 paramètres ; élève : 49,309,187
Réduction du générateur : 10.48 %


In [5]:
# --- 5. PERTES ET OPTIMISEURS ---
criterion_GAN = nn.MSELoss()
criterion_pixel = nn.L1Loss()

def distillation_loss(fake, y):
    # Superviser Y à plusieurs échelles, sans pénaliser le changement d'identité vers X1.
    loss = criterion_pixel(fake, y)
    for scale in (2, 4):
        loss = loss + criterion_pixel(F.avg_pool2d(fake, scale), F.avg_pool2d(y, scale))
    return loss / 3

def gradient_loss(fake, y):
    horizontal = criterion_pixel(fake[:, :, :, 1:] - fake[:, :, :, :-1],
                                 y[:, :, :, 1:] - y[:, :, :, :-1])
    vertical = criterion_pixel(fake[:, :, 1:, :] - fake[:, :, :-1, :],
                               y[:, :, 1:, :] - y[:, :, :-1, :])
    return (horizontal + vertical) / 2

def identity_loss(fake, identity):
    # Même ArcFace et même prétraitement que pour E1 dans l'application.
    arc_input = F.interpolate((fake - arc_mean) / arc_std, (112, 112), mode="nearest")
    predicted = F.normalize(arcface_model(arc_input), p=2, dim=1)
    return (1 - (predicted * identity).sum(dim=1)).mean()

optimizer_G = optim.Adam(student_model.parameters(), lr=0.0001, betas=(0.5, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=0.0001, betas=(0.5, 0.999))
scheduler_G = optim.lr_scheduler.ReduceLROnPlateau(optimizer_G, factor=0.5, patience=3)
print("Pertes : distillation vers Y, contours, identité X1 et adversarial progressif.")

Pertes : distillation vers Y, contours, identité X1 et adversarial progressif.


In [6]:
# --- 6. RÉSUMÉ DE L'ARCHITECTURE ET CONTRÔLE AVANT ENTRAÎNEMENT ---
# PyTorch prépare l'entraînement avec les pertes et optimiseurs définis plus haut.
# torch.compile() est une optimisation optionnelle, pas l'équivalent de compile() en Keras.
from html import escape
from IPython.display import HTML, display

def check_training_setup():
    required = ("student_model", "discriminator", "optimizer_G", "optimizer_D", "scheduler_G",
                "arcface_model", "distillation_loss", "gradient_loss", "identity_loss", "criterion_GAN")
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError("Exécuter les cellules d'initialisation et de pertes : " + ", ".join(missing))
    for name, model, optimizer in (("Élève", student_model, optimizer_G),
                                   ("Discriminateur", discriminator, optimizer_D)):
        expected = {id(p) for p in model.parameters()}
        registered = {id(p) for group in optimizer.param_groups for p in group["params"]}
        if not expected or expected != registered:
            raise RuntimeError(f"{name} : l'optimiseur ne correspond pas au modèle actuel ; relancer PERTES ET OPTIMISEURS")
    if not all(p.requires_grad for p in student_model.parameters()):
        raise RuntimeError("Des paramètres de l'élève sont gelés")
    if any(p.requires_grad for p in arcface_model.parameters()) or arcface_model.training:
        raise RuntimeError("ArcFace doit rester gelé et en mode eval()")
    if scheduler_G.optimizer is not optimizer_G:
        raise RuntimeError("Le scheduler ne correspond pas à l'optimiseur actuel")
    if not all(callable(loss) for loss in (distillation_loss, gradient_loss, identity_loss, criterion_GAN)):
        raise RuntimeError("Une fonction de perte n'est pas définie correctement")
    target = next(student_model.parameters()).device
    if target.type != torch.device(device).type or any(
            p.device != target or p.dtype != torch.float32
            for model in (student_model, discriminator, arcface_model) for p in model.parameters()):
        raise RuntimeError("Les modèles doivent partager le même périphérique et le type float32")

def architecture_summary(model, inputs, title):
    rows, handles = [], []
    modes = [(module, module.training) for module in model.modules()]

    def shapes(tensors):
        if torch.is_tensor(tensors):
            return str(tuple(tensors.shape))
        return ", ".join(str(tuple(t.shape)) for t in tensors if torch.is_tensor(t))

    def capture(name):
        def hook(module, args, result):
            rows.append((name, type(module).__name__, shapes(args), shapes(result),
                         f"{sum(p.numel() for p in module.parameters()):,}"))
        return hook

    try:
        for name, module in model.named_children():
            # Les blocs AdaIN sont appelés individuellement par le forward officiel.
            if name in {"BottleNeck", "model"}:
                for child_name, child in module.named_children():
                    handles.append(child.register_forward_hook(capture(f"{name}.{child_name}")))
            else:
                handles.append(module.register_forward_hook(capture(name)))
        model.eval()
        with torch.inference_mode():
            result = model(*inputs)
    finally:
        for handle in handles:
            handle.remove()
        for module, training in modes:
            module.training = training

    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    table = "<table><thead><tr>" + "".join(
        f"<th>{label}</th>" for label in ("Couche / bloc", "Type", "Entrée", "Sortie", "Paramètres"))
    table += "</tr></thead><tbody>" + "".join(
        "<tr>" + "".join(f"<td>{escape(value)}</td>" for value in row) + "</tr>" for row in rows)
    table += "</tbody></table>"
    display(HTML(f"<h3>{escape(title)}</h3>{table}"
                 f"<p><b>Total :</b> {total:,} — <b>Entraînables :</b> {trainable:,}"
                 f" — <b>Gelés :</b> {total - trainable:,} — <b>Poids float32 :</b> {total * 4 / 1024**2:.1f} Mio</p>"
                 f"<p>Entrées : {escape(shapes(inputs))} → Sortie : {escape(shapes(result))}</p>"
                 f"<details><summary>Afficher toutes les sous-couches</summary><pre>{escape(str(model))}</pre></details>"))
    return result

check_training_setup()
# Exemple synthétique : aucun parcours du dataset, aucune mise à jour des poids ou BatchNorm.
summary_device = next(student_model.parameters()).device
summary_x0 = torch.full((1, 3, 512, 512), 0.5, device=summary_device)
summary_identity = torch.zeros((1, 512), device=summary_device)
summary_identity[:, 0] = 1  # Norme L2 égale à 1, comme E1
summary_y = architecture_summary(student_model, (summary_x0, summary_identity), "Élève — SimSwap 512 allégé")
if (summary_y.shape != summary_x0.shape or not torch.isfinite(summary_y).all()
        or summary_y.min() < 0 or summary_y.max() > 1):
    raise RuntimeError("L'élève ne respecte pas le contrat RGB [1, 3, 512, 512] en [0, 1]")
summary_scores = architecture_summary(discriminator, (summary_x0, summary_y), "Discriminateur — entraînement uniquement")
if not torch.isfinite(summary_scores).all():
    raise RuntimeError("Le discriminateur produit des scores non finis")
del summary_x0, summary_identity, summary_y, summary_scores
print("Modèles, optimiseurs et pertes prêts. Passage avant et contrat SimSwap 512 vérifiés.")

Couche / bloc,Type,Entrée,Sortie,Paramètres
first_layer,Sequential,"(1, 3, 512, 512)","(1, 32, 512, 512)","4,800"
down0,Sequential,"(1, 32, 512, 512)","(1, 64, 256, 256)","18,624"
down1,Sequential,"(1, 64, 256, 256)","(1, 128, 128, 128)","74,112"
down2,Sequential,"(1, 128, 128, 128)","(1, 256, 64, 64)","295,680"
down3,Sequential,"(1, 256, 64, 64)","(1, 512, 32, 32)","1,181,184"
BottleNeck.0,ResnetBlock_Adain,"(1, 512, 32, 32), (1, 512)","(1, 512, 32, 32)","5,770,240"
BottleNeck.1,ResnetBlock_Adain,"(1, 512, 32, 32), (1, 512)","(1, 512, 32, 32)","5,770,240"
BottleNeck.2,ResnetBlock_Adain,"(1, 512, 32, 32), (1, 512)","(1, 512, 32, 32)","5,770,240"
BottleNeck.3,ResnetBlock_Adain,"(1, 512, 32, 32), (1, 512)","(1, 512, 32, 32)","5,770,240"
BottleNeck.4,ResnetBlock_Adain,"(1, 512, 32, 32), (1, 512)","(1, 512, 32, 32)","5,770,240"


Couche / bloc,Type,Entrée,Sortie,Paramètres
model.0,Conv2d,"(1, 6, 256, 256)","(1, 64, 128, 128)","6,208"
model.1,LeakyReLU,"(1, 64, 128, 128)","(1, 64, 128, 128)",0
model.2,Conv2d,"(1, 64, 128, 128)","(1, 128, 64, 64)","131,200"
model.3,InstanceNorm2d,"(1, 128, 64, 64)","(1, 128, 64, 64)",0
model.4,LeakyReLU,"(1, 128, 64, 64)","(1, 128, 64, 64)",0
model.5,Conv2d,"(1, 128, 64, 64)","(1, 256, 32, 32)","524,544"
model.6,InstanceNorm2d,"(1, 256, 32, 32)","(1, 256, 32, 32)",0
model.7,LeakyReLU,"(1, 256, 32, 32)","(1, 256, 32, 32)",0
model.8,Conv2d,"(1, 256, 32, 32)","(1, 1, 31, 31)","4,097"


Modèles, optimiseurs et pertes prêts. Passage avant et contrat SimSwap 512 vérifiés.


In [7]:
# --- 7. CHECKPOINTS COMPLETS ET REPRISE ---
import os

from html import escape
import sys
from IPython import get_ipython
from IPython.display import HTML, display

_progress_display = None
_progress_width = 0
_last_checkpoint_step = None

def training_progress(message, flush=True):
    # Une seule sortie Jupyter actualisee, commune au train, a la validation et aux checkpoints.
    global _progress_display, _progress_width
    line = str(message).replace("\n", " ").replace("\r", " ")
    if _last_checkpoint_step is not None:
        line += f" | Checkpoint step {_last_checkpoint_step}"
    shell = get_ipython()
    if shell is not None and getattr(shell, "kernel", None) is not None:
        content = HTML('<pre style="margin:0;white-space:pre;overflow-x:auto">' + escape(line) + '</pre>')
        if _progress_display is None:
            _progress_display = display(content, display_id=f"mfmc-training-{id(student_model)}")
        else:
            _progress_display.update(content)
    else:
        # Meme comportement sur une ligne si le code est lance depuis un terminal.
        sys.stdout.write("\r" + line.ljust(_progress_width))
        sys.stdout.flush()
        _progress_width = max(_progress_width, len(line))

checkpoint_dir = Path("outputs/gan_maison")
checkpoint_dir.mkdir(parents=True, exist_ok=True)
resume_training = True
checkpoint_every_steps = 500
num_epochs = 50
warmup_epochs = 5
early_stopping_start_epoch = warmup_epochs + 5 + 1  # Apres la montee adversariale (epoque 11)
patience_max = 8
if checkpoint_every_steps < 1:
    raise ValueError("La fréquence des checkpoints doit être positive")
checkpoint_path = checkpoint_dir / "training_last.pth"

def atomic_torch_save(data, path):
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("wb") as stream:
        torch.save(data, stream)
        stream.flush()
        os.fsync(stream.fileno())
    temporary.replace(path)

def atomic_json_save(data, path):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")
    temporary.replace(path)

def training_signature():
    return dict(manifest_sha256=sha256(manifest_path),
                train_size=len(dataloader.dataset), validation_size=len(val_loader.dataset),
                batch_size=dataloader.batch_size, drop_last=dataloader.drop_last,
                warmup_epochs=warmup_epochs, style_dim=student_model.style_dim,
                num_blocks=student_model.num_blocks, model_hashes=contract["model_hashes"],
                losses=dict(distillation=10., gradient=1., identity=1., adversarial=0.05))

def early_stopping_count(rows):
    monitored = [row["validation"] for row in rows
                 if row["epoch"] >= early_stopping_start_epoch]
    if not monitored:
        return 0
    best_index = min(range(len(monitored)), key=monitored.__getitem__)
    return len(monitored) - best_index - 1

def save_training_checkpoint():
    global _last_checkpoint_step
    # Appeler uniquement entre deux batches complets ou à une frontière d'époque.
    state = dict(version=1, signature=run_signature,
                 student=student_model.state_dict(), discriminator=discriminator.state_dict(),
                 optimizer_G=optimizer_G.state_dict(), optimizer_D=optimizer_D.state_dict(),
                 scheduler_G=scheduler_G.state_dict(),
                 epoch=epoch, next_batch=next_batch, global_step=global_step,
                 epoch_order=epoch_order, shuffle_rng=shuffle_generator.get_state(),
                 totals=totals, seen=seen, history=history,
                 best_val_loss=best_val_loss, patience_counter=patience_counter,
                 rng_python=random.getstate(), rng_numpy=np.random.get_state(),
                 rng_torch=torch.get_rng_state(),
                 rng_cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [])
    atomic_torch_save(state, checkpoint_path)
    _last_checkpoint_step = global_step
    training_progress(f"Checkpoint complet : {global_step} steps ; époque {epoch + 1}, prochain batch {next_batch}")

def restore_training_checkpoint():
    # Checkpoint créé localement : inclut les états Python/NumPy, au-delà des seuls poids.
    state = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    saved_signature = state.get("signature")
    if saved_signature != run_signature:
        # Migration NPY -> PNG : accepter uniquement le manifeste converti et verifie.
        migration_path = dataset_dir / "Y_conversion.json"
        if migration_path.exists() and isinstance(saved_signature, dict):
            migration = json.loads(migration_path.read_text(encoding="utf-8"))
            if (migration.get("complete")
                    and saved_signature.get("manifest_sha256") == migration.get("old_manifest_sha256")
                    and run_signature["manifest_sha256"] == migration.get("new_manifest_sha256")):
                saved_signature = dict(saved_signature, manifest_sha256=run_signature["manifest_sha256"])
                if saved_signature == run_signature:
                    training_progress("Reprise apres migration verifiee des Y en PNG uint8.")
    if state.get("version") != 1 or saved_signature != run_signature:
        raise ValueError("Checkpoint incompatible avec le dataset, le batch size ou les paramètres d'entraînement")
    student_model.load_state_dict(state["student"], strict=True)
    discriminator.load_state_dict(state["discriminator"], strict=True)
    optimizer_G.load_state_dict(state["optimizer_G"])
    optimizer_D.load_state_dict(state["optimizer_D"])
    scheduler_G.load_state_dict(state["scheduler_G"])
    shuffle_generator.set_state(state["shuffle_rng"])
    random.setstate(state["rng_python"])
    np.random.set_state(state["rng_numpy"])
    torch.set_rng_state(state["rng_torch"])
    if torch.cuda.is_available() and state["rng_cuda"]:
        torch.cuda.set_rng_state_all(state["rng_cuda"])
    return state

print(f"Reprise : {resume_training} | Checkpoint tous les {checkpoint_every_steps} steps | {checkpoint_path.resolve()} | Existe : {checkpoint_path.exists()}")

Reprise : True | Checkpoint tous les 500 steps | H:\M2-Projet-Image-GROUPE-1\code\ImplementationMFMC\outputs\gan_maison\training_last.pth | Existe : True


In [8]:
# --- 8. ENTRAÎNEMENT ET VALIDATION AVEC REPRISE ---
from time import perf_counter

_progress_display = None
_progress_width = 0
_last_checkpoint_step = None

if "check_training_setup" not in globals():
    raise RuntimeError("Exécuter la cellule RÉSUMÉ DE L'ARCHITECTURE avant l'entraînement")
if "restore_training_checkpoint" not in globals():
    raise RuntimeError("Exécuter la cellule CHECKPOINTS COMPLETS ET REPRISE avant l'entraînement")
if "training_progress" not in globals() or "early_stopping_count" not in globals():
    raise RuntimeError("Reexecuter CHECKPOINTS COMPLETS ET REPRISE avant de lancer cette cellule")
check_training_setup()
if dataloader.batch_size is None:
    raise ValueError("Un batch size explicite est nécessaire pour la reprise")
run_signature = training_signature()
shuffle_generator = torch.Generator().manual_seed(42)
epoch, next_batch, global_step = 0, 0, 0
epoch_order = None
totals = dict(G=0., D=0., distill=0., identity=0.)
seen = 0
best_val_loss = float("inf")
patience_counter = 0
history = []

if checkpoint_path.exists():
    if not resume_training:
        raise FileExistsError("Un checkpoint existe : activer resume_training ou choisir un autre checkpoint_dir")
    state = restore_training_checkpoint()
    epoch, next_batch, global_step = state["epoch"], state["next_batch"], state["global_step"]
    epoch_order, totals, seen = state["epoch_order"], state["totals"], state["seen"]
    history = state["history"]
    best_val_loss = state["best_val_loss"]
    patience_counter = early_stopping_count(history)
    del state
    _last_checkpoint_step = global_step
    atomic_json_save(history, checkpoint_dir / "history.json")
    training_progress(f"Reprise : époque {epoch + 1}/{num_epochs}, batch {next_batch}/{len(dataloader)}, step global {global_step}")
else:
    # Une interruption avant les 500 premiers steps pourra déjà repartir de cet état.
    save_training_checkpoint()

training_progress("Lancement de l'entraînement...")
active_iterator = None
try:
    while epoch < num_epochs and patience_counter < patience_max:
        student_model.train()
        discriminator.train()
        if epoch_order is None:
            epoch_order = torch.randperm(len(dataloader.dataset), generator=shuffle_generator)
        # Sauter les indices déjà traités AVANT le chargement : aucun ancien X0/Y à relire.
        remaining = epoch_order[next_batch * dataloader.batch_size:].tolist()
        epoch_loader = DataLoader(dataloader.dataset, batch_size=dataloader.batch_size,
                                  sampler=remaining, drop_last=dataloader.drop_last,
                                  num_workers=0, pin_memory=dataloader.pin_memory,
                                  generator=torch.Generator().manual_seed(0))
        adversarial_weight = 0.05 * min(1., max(0., (epoch - warmup_epochs + 1) / 5))

        active_iterator = iter_prepared_batches(epoch_loader)
        for batch_idx, (x0, identity, y) in enumerate(active_iterator, start=next_batch):
            x0, identity, y = x0.to(device, non_blocking=True), identity.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimizer_G.zero_grad()
            fake = student_model(x0, identity)

            # PHASE 1 : LE DISCRIMINATEUR COMPARE Y ET LA SORTIE DE L'ÉLÈVE
            loss_D = torch.zeros((), device=device)
            if adversarial_weight > 0:
                discriminator.requires_grad_(True)
                optimizer_D.zero_grad()
                pred_real = discriminator(x0, y)
                pred_fake = discriminator(x0, fake.detach())
                loss_D = (criterion_GAN(pred_real, torch.ones_like(pred_real))
                          + criterion_GAN(pred_fake, torch.zeros_like(pred_fake))) / 2
                if not torch.isfinite(loss_D):
                    raise FloatingPointError("Perte discriminateur non finie")
                loss_D.backward()
                optimizer_D.step()

            # PHASE 2 : DISTILLATION ET IDENTITÉ DU DONNEUR
            discriminator.requires_grad_(False)
            loss_distill = distillation_loss(fake, y)
            loss_id = identity_loss(fake, identity)
            loss_G = 10 * loss_distill + gradient_loss(fake, y) + loss_id
            if adversarial_weight > 0:
                pred_fake_for_G = discriminator(x0, fake)
                loss_G = loss_G + adversarial_weight * criterion_GAN(
                    pred_fake_for_G, torch.ones_like(pred_fake_for_G))
            if not torch.isfinite(loss_G):
                raise FloatingPointError("Perte générateur non finie")
            loss_G.backward()
            nn.utils.clip_grad_norm_(student_model.parameters(), max_norm=5., error_if_nonfinite=True)
            optimizer_G.step()

            count = x0.shape[0]
            seen += count
            # Une seule copie des quatre scalaires au lieu de quatre appels item().
            metrics = torch.stack((loss_G.detach(), loss_D.detach(),
                                   loss_distill.detach(), loss_id.detach())).cpu().tolist()
            for name, value in zip(("G", "D", "distill", "identity"), metrics):
                totals[name] += value * count
            next_batch = batch_idx + 1
            global_step += 1
            if batch_idx % 25 == 0:
                training_progress(f"Epoch {epoch + 1}/{num_epochs} | Batch {batch_idx}/{len(dataloader)} "
                      f"| G={metrics[0]:.4f} | D={metrics[1]:.4f}")
            if global_step % checkpoint_every_steps == 0:
                save_training_checkpoint()

        # Conserver tous les batches train avant la validation, même si elle est interrompue.
        save_training_checkpoint()
        student_model.eval()
        val_total, val_seen = 0., 0
        val_batches = len(val_loader)
        val_started = perf_counter()
        training_progress(f"Validation | Epoch {epoch + 1}/{num_epochs} | Batch 0/{val_batches}", flush=True)
        with torch.inference_mode():
            active_iterator = iter_prepared_batches(val_loader)
            for val_batch, (x0, identity, y) in enumerate(active_iterator, start=1):
                x0, identity, y = x0.to(device, non_blocking=True), identity.to(device, non_blocking=True), y.to(device, non_blocking=True)
                prediction = student_model(x0, identity)
                val_total += distillation_loss(prediction, y).item() * x0.shape[0]
                val_seen += x0.shape[0]
                if val_batch == 1 or val_batch % 25 == 0 or val_batch == val_batches:
                    elapsed = perf_counter() - val_started
                    remaining = elapsed / val_batch * (val_batches - val_batch)
                    training_progress(f"Validation | Epoch {epoch + 1}/{num_epochs} | Batch {val_batch}/{val_batches} "
                          f"({100 * val_batch / val_batches:.1f} %) | L1={val_total / val_seen:.5f} "
                          f"| Temps {elapsed / 60:.1f} min | Reste ~{remaining / 60:.1f} min", flush=True)
        val_loss = val_total / val_seen
        if not np.isfinite(val_loss):
            raise FloatingPointError("Perte validation non finie")
        scheduler_G.step(val_loss)
        row = dict(epoch=epoch + 1, **{name: value / seen for name, value in totals.items()},
                   validation=val_loss, lr=optimizer_G.param_groups[0]["lr"])
        history.append(row)
        patience_counter = early_stopping_count(history)
        training_progress(f"Epoch {epoch + 1} | distillation train={row['distill']:.5f} | validation={val_loss:.5f}")

        # EXPORT D'INFÉRENCE : LES MEILLEURS POIDS RESTENT SÉPARÉS DU CHECKPOINT DE REPRISE.
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            atomic_torch_save(student_model.state_dict(), checkpoint_dir / "lite_generator_best.pth")
            model_config = dict(architecture="CustomLiteGenerator / SimSwap512", style_dim=512,
                                num_blocks=8, deep=False, crop_size=512, alignment="ffhq",
                                color="RGB", image_range=[0, 1], identity="SimSwap ArcFace L2",
                                teacher_params=teacher_params, student_params=student_params,
                                epoch=epoch + 1, validation=val_loss,
                                model_hashes=contract["model_hashes"], manifest_sha256=run_signature["manifest_sha256"])
            atomic_json_save(model_config, checkpoint_dir / "lite_generator_best.json")
            training_progress("Meilleur élève enregistré !")

        epoch += 1
        next_batch, epoch_order, seen = 0, None, 0
        totals = dict(G=0., D=0., distill=0., identity=0.)
        save_training_checkpoint()
        atomic_json_save(history, checkpoint_dir / "history.json")
        if patience_counter >= patience_max:
            training_progress("Arrêt anticipé : la validation ne progresse plus.")
except KeyboardInterrupt:
    # Un arrêt peut couper un batch entre les mises à jour de D et G : ne pas sauver cet état partiel.
    training_progress(f"Entraînement interrompu. Dernier checkpoint complet conservé : {checkpoint_path}")
    training_progress("Relancer cette cellule pour reprendre depuis ce checkpoint (environ 500 steps au maximum à rejouer).")
else:
    training_progress("Entraînement terminé !")
finally:
    if active_iterator is not None:
        active_iterator.close()


In [ ]:
# --- 7. COURBES ET CONTRÔLE DU MEILLEUR ÉLÈVE ---
student_model.load_state_dict(torch.load(checkpoint_dir / "lite_generator_best.pth",
                                        map_location=device, weights_only=True), strict=True)
student_model.eval()
plt.figure(figsize=(8, 4))
plt.plot([row["epoch"] for row in history], [row["distill"] for row in history], label="Train vers Y")
plt.plot([row["epoch"] for row in history], [row["validation"] for row in history], label="Validation vers Y")
plt.xlabel("Epoch")
plt.ylabel("L1 multi-échelle")
plt.legend()
plt.show()

x0, identity, y = next(iter(val_loader))
# Le portrait X1 est lu uniquement pour cette visualisation.
donor_rgb = load_image(dataset_dir / val_records[0]["X1"])[..., ::-1]
x1 = torch.from_numpy(donor_rgb.transpose(2, 0, 1).copy()).float() / 255
with torch.inference_mode():
    prediction = student_model(x0.to(device), identity.to(device)).cpu()
assert prediction.shape == x0.shape and torch.isfinite(prediction).all()
assert prediction.min() >= 0 and prediction.max() <= 1
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, title, image in zip(axes, ("X0", "X1 donneur", "Y maître", "Élève"),
                            (x0[0], x1, y[0], prediction[0])):
    ax.imshow(image.permute(1, 2, 0).numpy())
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()
print("Contrat vérifié : générateur (RGB [0, 1], identité SimSwap L2) -> RGB [0, 1].")